In [7]:
import torch
from torch import nn

In [8]:
def compute_advantage(rewards):
  meanv = rewards.mean()
  baseline = meanv.detach()
  advantage = rewards - baseline
  return advantage

In [9]:
def policy_entropy(logits):
  dist = torch.distributions.Categorical(logits=logits)
  ts = dist.entropy()
  return ts.mean()

In [10]:
def categorical_kl(old_logits,new_logits):
  dist_old = old_logits
  dist_new = new_logits
  klloss = torch.distributions.kl_divergence(dist_old,dist_new)
  return klloss.mean()

In [11]:
class PolicyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(2,2),
        nn.ReLU(),
        nn.Linear(2,2)
    )
  def forward(self,x):
    return self.net(x)

In [12]:
def sample_context(batch_size):
  return torch.randn(batch_size,2)

In [13]:
def get_reward(context,action):
  result = (action == torch.where(context[:,0] > context[:,1],0,1))
  return result.float()

In [14]:
policy = PolicyNet()
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-2)
entropy_coef = 0.01

for step in range(300):
  contexts = sample_context(32)
  old_logits = policy(contexts).detach()

  logits = policy(contexts)
  dist = torch.distributions.Categorical(logits=logits)
  actions = dist.sample()
  log_probs = dist.log_prob(actions)

  rewards = get_reward(contexts, actions)
  advantages = compute_advantage(rewards)


  entropy = policy_entropy(logits)
  pg_loss = -(log_probs * advantages).mean()
  loss = pg_loss - entropy_coef * entropy


  optimizer.zero_grad()
  loss.backward()
  optimizer.step()


  if (step + 1) % 50 == 0:

      new_logits = policy(contexts)
      kl = categorical_kl(old_logits, new_logits)

      print(f"Step {step+1}: Reward={rewards.mean().item():.2f}, "
            f"Entropy={entropy.item():.3f}, KL={kl.item():.4f}")

NotImplementedError: No KL(p || q) is implemented for p type Tensor and q type Tensor